# Laya V3.2 — Can a MaxPool U-Net and Pretrained Q/K Landmark Memory Replace Full Attention at Layer 15?

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V32_Layer15_UNet_MaxPool_LandmarkMemory_Colab.ipynb)

## Finding from V3.1
On its held-out audit, V3.1 local cellular achieved cosine **0.7514**, long-dilated **0.7786**, and fused **0.7823**, yet the fusion's layer took **8.04 ms** vs pretrained full attention **1.45 ms**. Its 500-decision agreement was **0.794** with 12.4 percentage points lower gold accuracy. The attached JSON used 600 train, 100 probe, and 100 audit cases and only 288 unique cached training decisions.

**Crucial structural diagnosis:** V3.1 explicitly removed the teacher's pretrained **Q/K content routing** and **RoPE**: it copied only pretrained Wv and Wo. Merely enlarging the convolutional receptive field to 511 tokens did not restore content-dependent retrieval. MaxPool/U-Net can recover multiscale salience with skip connections, but still cannot dynamically select an arbitrary remote token by its key.

## Four controlled candidates
1. **unet_avg:** 1D U-Net with *masked average pooling* + full-resolution skip connections; no attention.
2. **unet_max:** identical U-Net with *masked max pooling* + full-resolution skip connections; no attention.
3. **landmark_qk:** compressed memory built from frozen pretrained **RoPE Q,K,V**; block-mean, learned soft-salience, and max-norm token landmarks. **Compressed cross-attention, not attention-free.**
4. **unet_max_qk:** max-pool U-Net **plus** compressed Q/K landmark memory, mixed using a per-token learned fusion gate.

All use the frozen pretrained Wqkv/Wo. Landmark memory is **3 landmarks per 16 tokens**; for length 512, ~96 memory keys instead of 512, but compressed memory remains content-addressed attention with \(\mathcal O(T\lceil T/16\rceil)\) scores. U-Net uses 4 down/up-sampling stages and high-resolution skip connections, plus dilated coarse bottleneck operators. The non-attention U-Net branch has **over-256-token theoretical receptive field** via the coarse path; actual effective RF and speed are measured, not assumed.

## Protocol
- Reconstruct the exact native teacher call, audit max pooling on negative/padded data, verify finite gradients and **identical-shape masked-padding invariance** before any training.
- **Screen: 160 updates per candidate**, identical teacher cache and case-disjoint train/probe/audit. No implicit endless 1,200-step refinement.
- **Optional refine: 320 updates, off by default**, only if probe cosine exceeds the predeclared 0.80 *GO* threshold. This gate is for deciding whether to invest more GPU time, not a claim of acceptable replacement fidelity.
- Store each trained checkpoint *before* evaluating e2e and latency; resume completed candidates.
- Final metrics on the same held-out audit split: local cosine, NMSE, decision agreement, teacher/student gold accuracy, JS divergence, token-level layer latency and full-model latency.
- Strict production-replacement gates: cosine≥0.95, NMSE≤0.10, agreement≥0.95, mean JS≤0.02, layer speedup >1.0.
- The official external TEST split is untouched. The audit split is a report-only holdout, never used to choose the winner.
- Important: **U-Net with max pooling alone is not mathematically equivalent to global attention**. The experiment specifically isolates whether reintroducing pretrained query/key retrieval closes the remaining gap.

**Fair inference comparison:** U-Net-only models evaluate only the pretrained V projection, whereas landmark-memory models evaluate full pretrained Q/K/V (needed for content retrieval). This avoids charging unused Q/K projection cost to the attention-free baselines.


In [ ]:
#@title 1. Install exact training dependencies
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Imports, frozen Laya teacher and experiment configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds
from tinycenn_lm.laya_lab.core import _valid_tokens
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question,
)
from tinycenn_lm.decision_training import require_finite

SEED = 2032
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15 #@param {type:"integer"}
TARGET_LAYERS = (LAYER,)
TRAIN_CASES = 600 #@param {type:"integer"}
DEV_CASES = 200 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
STEPS = 160 #@param {type:"integer"}
REFINE_STEPS = 320 #@param {type:"integer"}
REFINE_TOP_K = 1
RUN_REFINE = False #@param {type:"boolean"}
REFINE_GO_COSINE = 0.80
MEMORY_STRIDE = 16 #@param {type:"integer"}
UNET_WIDTH = 256 #@param {type:"integer"} #@param {type:"integer"}
TRAIN_CACHE_BATCHES = 96 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 16 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 16 #@param {type:"integer"}
CHECK_EVERY = 100 #@param {type:"integer"}
OUTPUT_DIR = Path("/content/Laya_V32_UNet_MaxPool_Memory")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert STEPS >= 1 and DEV_CASES >= 4, "Need at least one successful update and a meaningful held-out split"
assert REFINE_TOP_K >= 1 and TRAIN_CACHE_BATCHES > 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Use a CUDA GPU runtime in Google Colab.")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL, allow_patterns=[
        "rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
assert LAYER in full_layers, (LAYER, full_layers)
print("GPU:", torch.cuda.get_device_name(0), "AMP:", amp_dtype)
print("Source:", SOURCE_MODEL)
print("Target:", LAYER, "all full attention layers:", full_layers)
print("V3.2 screen:", STEPS, "refine:", REFINE_STEPS, "enabled:", RUN_REFINE)
print("Memory stride:", MEMORY_STRIDE, "U-Net width:", UNET_WIDTH,"official test used: NO")


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_ids=entry["call_kwargs"].get("position_ids"),
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V3.2 local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items) >= PROBE_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce PROBE_CACHE_BATCHES"
assert len(audit_items) >= AUDIT_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce AUDIT_CACHE_BATCHES"
print("official test remains untouched")


## A. Mathematics: what U-Net fixes, and what only Q/K retrieval fixes

**Masked max pooling** at scale 2 retains each channel's extreme activation, but it can discard location and almost everything about non-maximal features. **Skip connections** recover high-resolution token identity; masked pooling avoids padding zeros being selected over negative valid activations. Average pooling provides a same-capacity ablation.

For the U-Net coarse branch with stride \(16\) and kernel-size-5 convolutions with dilation \(2,4,8\) on the bottleneck grid, the coarse bottleneck alone has a theoretical receptive span \(1 + 4\cdot(2+4+8)\cdot16=897\) tokens (larger than our 512-token maximum, truncated at sequence boundaries). This is **not** proof of effective access to arbitrary distant token identities.

The missing mechanism is the pretrained teacher's key-dependent read:

\[
y_i = W_O\left[\sum_j\operatorname{softmax}_j(q_i^\top k_j/\sqrt{d_h})v_j\right].
\]

The compressed landmark branch preserves pretrained RoPE-rotated Q and K. In each 16-token block it stores three K/V landmarks: masked mean, a learned soft-salience weighted token, and the **same max-key-norm token** (keeping its true matching K and V, unlike channelwise max pooling). It then evaluates scaled-dot-product attention from *each original query* to this compressed memory:

\[
\widetilde y_i=W_O\sum_{m=1}^{M}
\operatorname{softmax}_m(q_i^\top\widetilde k_m/\sqrt{d_h})\,\widetilde v_m,\qquad
M=3\lceil T/16\rceil.
\]

This is **not attention-free**; it reduces key count and tests whether selective remote retrieval is the missing ingredient. Learned dynamic landmark selection is differentiable via a soft-salience pathway; the max-norm hard-index branch provides an additional exact original token. The pure max-pool U-Net remains an explicitly attention-free comparison.

We do **not** assume that grouped K/V faithfully approximate the original softmax normalization or that 96 landmarks suffice: the notebook measures real layer errors and a compressed-landmark teacher baseline before training.


In [ ]:
#@title 4. Mask-safe 1D U-Net + pretrained RoPE landmark memory (four ablations)
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb

UNET_LEVELS = 4
UNET_BOTTLENECK_DILATIONS = (2,4,8)
CANDIDATE_NAMES = ("unet_avg","unet_max","landmark_qk","unet_max_qk")
CANDIDATE_INFO = {
    name: {"lr_start": 0.0015, "lr_end": 0.00012,
           "label": name.replace("_"," ").title()}
    for name in CANDIDATE_NAMES
}
assert MEMORY_STRIDE >= 4 and MEMORY_STRIDE <= 64
assert UNET_WIDTH >= 64 and UNET_WIDTH % 8 == 0
assert UNET_LEVELS == 4

def masked_pool2(x, mask, kind):
    """Masked stride-2 pooling; a padding zero must never beat a negative max."""
    assert kind in ("avg","max")
    b,c,t=x.shape
    extra=t%2
    xp=F.pad(x,(0,extra)).reshape(b,c,(t+extra)//2,2)
    mp=F.pad(mask,(0,extra)).reshape(b,1,(t+extra)//2,2).bool()
    count=mp.sum(-1)
    valid=(count>0)
    if kind=="avg":
        pooled=(xp*mp.to(x.dtype)).sum(-1)/count.to(x.dtype).clamp_min(1)
    else:
        pooled=xp.masked_fill(~mp,torch.finfo(x.dtype).min).amax(-1)
    pooled=torch.where(valid,pooled,torch.zeros_like(pooled))
    return pooled,valid.to(x.dtype)

class DWRes(nn.Module):
    """Lightweight masked gated depthwise residual conv."""
    def __init__(self,width,dilation=1):
        super().__init__()
        self.dw=nn.Conv1d(width,width,kernel_size=5,padding=2*dilation,
                          dilation=dilation,groups=width,bias=False)
        self.pw=nn.Conv1d(width,2*width,kernel_size=1)
        self.log_gain=nn.Parameter(torch.tensor(-2.2))
        nn.init.zeros_(self.dw.weight)
        with torch.no_grad(): self.dw.weight[:,0,2].fill_(0.95)
        nn.init.zeros_(self.pw.bias)

    def forward(self,x,mask):
        a,b=self.pw(self.dw(x*mask)).chunk(2,dim=1)
        return (x + self.log_gain.exp()*torch.tanh(a)*torch.sigmoid(b))*mask

class MaskedUNet1D(nn.Module):
    """Four-level 1D U-Net; full-resolution encoder skips restore token detail."""
    def __init__(self,width=UNET_WIDTH,pool_kind="max"):
        super().__init__()
        self.pool_kind=pool_kind
        self.enc=nn.ModuleList([DWRes(width) for _ in range(UNET_LEVELS)])
        self.bottleneck=nn.ModuleList([
            DWRes(width,dilation=d) for d in UNET_BOTTLENECK_DILATIONS
        ])
        self.merge=nn.ModuleList([
            nn.Conv1d(2*width,width,1) for _ in range(UNET_LEVELS)
        ])
        self.dec=nn.ModuleList([DWRes(width) for _ in range(UNET_LEVELS)])
        for m in self.merge:
            nn.init.zeros_(m.bias)
            nn.init.normal_(m.weight,std=0.015)
        self.out_norm=nn.LayerNorm(width)

    def forward(self,x,mask):
        h=x*mask
        skips=[]
        masks=[]
        for stage in self.enc:
            h=stage(h,mask)
            skips.append(h)
            masks.append(mask)
            h,mask=masked_pool2(h,mask,self.pool_kind)
        for stage in self.bottleneck:
            h=stage(h,mask)
        for skip,sm,merge,stage in zip(
            reversed(skips),reversed(masks),reversed(self.merge),reversed(self.dec)
        ):
            h=F.interpolate(h,size=skip.shape[-1],mode="nearest")
            h=merge(torch.cat([h,skip],dim=1))
            h=stage(h,sm)
            mask=sm
        return self.out_norm(h.transpose(1,2)).transpose(1,2)*mask

class LandmarkMemory(nn.Module):
    """Pretrained Q/K/V RoPE with 3 block landmarks; no N-by-N token matrix."""
    def __init__(self,hidden,heads,head_dim,stride=MEMORY_STRIDE):
        super().__init__()
        self.hidden=hidden
        self.heads=heads
        self.head_dim=head_dim
        self.stride=stride
        self.salience=nn.Linear(hidden,1)
        nn.init.normal_(self.salience.weight,std=0.002)
        nn.init.zeros_(self.salience.bias)
        self.salience_temp=nn.Parameter(torch.tensor(0.0))
        self.q_log_scale=nn.Parameter(torch.tensor(0.0))

    def forward(self,q,k,v,hidden,valid):
        # q/k/v: [B,H,T,d], pretrained RoPE already applied to q and k.
        b,h,t,d=q.shape
        stride=self.stride
        extra=(-t)%stride
        groups=(t+extra)//stride
        # Work with [B,T,H,d] so block pooling preserves whole K/V vectors.
        kb=k.transpose(1,2)
        vb=v.transpose(1,2)
        if extra:
            kb=torch.cat([kb,kb.new_zeros((b,extra,h,d))],dim=1)
            vb=torch.cat([vb,vb.new_zeros((b,extra,h,d))],dim=1)
            xp=torch.cat([hidden,hidden.new_zeros((b,extra,hidden.shape[-1]))],dim=1)
            vp=torch.cat([valid,torch.zeros((b,extra),device=valid.device,dtype=torch.bool)],dim=1)
        else:
            xp=hidden
            vp=valid
        kg=kb.reshape(b,groups,stride,h,d)
        vg=vb.reshape(b,groups,stride,h,d)
        mask=vp.reshape(b,groups,stride)
        m=mask[...,None,None].to(kg.dtype)
        count=mask.sum(-1).clamp_min(1)[...,None,None].to(kg.dtype)
        km=(kg*m).sum(2)/count
        vm=(vg*m).sum(2)/count

        # Learnable differentiable soft token selection inside each block.
        score=self.salience(xp).reshape(b,groups,stride).float()
        score=score/self.salience_temp.clamp(-2,2).exp()
        score=score.masked_fill(~mask,-1e4)
        p=torch.softmax(score,dim=-1)*mask.float()
        p=p/p.sum(-1,keepdim=True).clamp_min(1e-8)
        soft=p[...,None,None].to(kg.dtype)
        ks=(kg*soft).sum(2)
        vs=(vg*soft).sum(2)

        # Hard landmark is a WHOLE original token, not a synthetic max vector.
        sal=kg.float().square().sum(-1).mean(-1)
        index=sal.masked_fill(~mask,-1e20).argmax(-1)
        gather_idx=index[...,None,None,None].expand(b,groups,1,h,d)
        kh=kg.gather(2,gather_idx).squeeze(2)
        vh=vg.gather(2,gather_idx).squeeze(2)

        keys=torch.stack((km,ks,kh),dim=2).reshape(b,groups*3,h,d)
        values=torch.stack((vm,vs,vh),dim=2).reshape(b,groups*3,h,d)
        keys=keys.transpose(1,2).contiguous()
        values=values.transpose(1,2).contiguous()
        memory_valid=mask.any(-1).repeat_interleave(3,dim=1)
        attn_mask=memory_valid[:,None,None,:]
        qs=q*self.q_log_scale.clamp(-1,1).exp()
        # Memory attention is T x M, with M = 3*ceil(T/stride).
        result=F.scaled_dot_product_attention(
            qs,keys,values,attn_mask=attn_mask,dropout_p=0.0
        )
        result=result.transpose(1,2).contiguous().reshape(b,t,h*d)
        return result*valid[:,:,None].to(result.dtype)

class Layer15UNetLandmark(nn.Module):
    """Drop-in ModernBERT attention replacement; retains native frozen QKV+RoPE."""
    def __init__(self,original,mode,width=UNET_WIDTH,stride=MEMORY_STRIDE):
        super().__init__()
        if mode not in CANDIDATE_NAMES: raise ValueError(mode)
        self.mode=mode
        self.config=original.config
        self.width=width
        self.stride=stride
        self.hidden_size=int(original.config.hidden_size)
        self.heads=int(original.num_heads)
        self.head_dim=int(original.head_dim)
        d=self.hidden_size
        self.Wqkv=copy.deepcopy(original.Wqkv).requires_grad_(False)
        self.Wo=copy.deepcopy(original.Wo).requires_grad_(False)
        self.rotary_emb=copy.deepcopy(original.rotary_emb).requires_grad_(False)
        self.out_drop=copy.deepcopy(original.out_drop).requires_grad_(False)
        self.has_unet=mode!="landmark_qk"
        self.has_memory=mode in ("landmark_qk","unet_max_qk")
        if self.has_unet:
            self.input_norm=nn.LayerNorm(d)
            self.down=nn.Linear(d,width,bias=False)
            self.unet=MaskedUNet1D(width,pool_kind="avg" if mode=="unet_avg" else "max")
            self.up=nn.Linear(width,d,bias=False)
        if self.has_memory:
            self.memory=LandmarkMemory(d,self.heads,self.head_dim,stride=stride)
        if self.has_unet and self.has_memory:
            self.fuse_gate=nn.Linear(d,1)
            nn.init.zeros_(self.fuse_gate.weight)
            nn.init.zeros_(self.fuse_gate.bias)
        self.anchor_logit=nn.Parameter(torch.tensor(-1.2))
        self.branch_logit=nn.Parameter(torch.tensor(1.2))

    def forward(self,hidden_states,attention_mask=None,position_ids=None,**kwargs):
        b,t,d=hidden_states.shape
        valid=_valid_tokens(attention_mask,hidden_states)
        m=valid[:,:,None].to(hidden_states.dtype)
        # U-Net-only ablations compute V directly, avoiding the cost of Q/K.
        # Memory candidates compute full pretrained QKV and retain native RoPE.
        if self.has_memory:
            qkv=self.Wqkv(hidden_states).view(b,t,3,self.heads,self.head_dim)
            q,k,v=qkv.transpose(3,1).unbind(dim=2)
            vflat=v.transpose(1,2).contiguous().reshape(b,t,d)*m
        else:
            bias=None if self.Wqkv.bias is None else self.Wqkv.bias[2*d:3*d]
            vflat=F.linear(hidden_states,self.Wqkv.weight[2*d:3*d],bias)*m
        unet_out=None
        memory_out=None
        if self.has_unet:
            u=self.down(self.input_norm(vflat)).transpose(1,2)
            out=self.unet(u,valid[:,None,:].to(u.dtype)).transpose(1,2)
            unet_out=self.up(out)*m
        if self.has_memory:
            if position_ids is None:
                position_ids=torch.arange(t,device=hidden_states.device)[None,:]
            cos,sin=self.rotary_emb(qkv,position_ids=position_ids)
            qr,kr=apply_rotary_pos_emb(q,k,cos,sin)
            memory_out=self.memory(qr,kr,v,hidden_states,valid)
        if self.has_unet and self.has_memory:
            gate=torch.sigmoid(self.fuse_gate(hidden_states))
            branch=gate*unet_out+(1.0-gate)*memory_out
        elif self.has_memory:
            branch=memory_out
        else:
            branch=unet_out
        a=torch.sigmoid(self.anchor_logit)
        g=torch.sigmoid(self.branch_logit)
        mixed=a*vflat+g*branch
        result=self.Wo(mixed)*m
        return self.out_drop(result),None

    def config_dict(self):
        return dict(architecture="v32_1d_unet_landmarks",mode=self.mode,
                    width=self.width,pooling=("avg" if self.mode=="unet_avg" else "max")
                    if self.has_unet else "none",
                    levels=UNET_LEVELS,memory_stride=self.stride if self.has_memory else None,
                    landmarks_per_group=3 if self.has_memory else 0,
                    uses_pretrained_rope_qk=self.has_memory,
                    attention_free=not self.has_memory)

def build_candidate(name,original):
    return Layer15UNetLandmark(original,name).to(device)

def candidate_parameters(model):
    return [p for p in model.parameters() if p.requires_grad]

print("Architectures:")
for n in CANDIDATE_NAMES:
    m=build_candidate(n,teacher.encoder.layers[LAYER].attn)
    print(f" {n:<20} trainable={sum(p.numel() for p in candidate_parameters(m)):,}"
          f" | uses QK retrieval={m.has_memory}")
    del m
torch.cuda.empty_cache()


In [ ]:
#@title 5. Strict preflight: teacher replay, maxpool correctness, padding leaks, gradients
teacher_attn=teacher.encoder.layers[LAYER].attn
probe_first=batch(probe_items,0,BATCH)
sample=attn_io(teacher,LAYER,probe_first)
with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
    teacher_replay=_attention_output_tensor(
        teacher_attn(*sample["call_args"],**sample["call_kwargs"])
    )
assert torch.allclose(teacher_replay.float(),sample["y"].float(),atol=1e-2,rtol=1e-2)
print("Teacher native SDPA replay: PASS")
# Exact numerical test: padded 0 must not replace negative values in max pooling.
px=torch.tensor([[[-3.,-7.,-5.,100.]]],device=device)
pm=torch.tensor([[[1.,1.,1.,0.]]],device=device)
po,valid_p=masked_pool2(px,pm,"max")
assert torch.allclose(po,torch.tensor([[[-3.,-5.]]],device=device))
assert valid_p.bool().all()
print("Masked negative-value MaxPool: PASS")

def preflight_model(name):
    model=build_candidate(name,teacher_attn)
    model.train()
    with torch.autocast(device_type="cuda",dtype=amp_dtype):
        pred,_=model(sample["x"],attention_mask=sample["mask"],
                     position_ids=sample["call_kwargs"].get("position_ids"))
        valid=probe_first["attention_mask"].bool()
        loss=(pred.float()[valid]-sample["y"].float()[valid]).square().mean()
    require_finite(pred,name+" first forward")
    require_finite(loss,name+" first loss")
    grads=torch.autograd.grad(loss,candidate_parameters(model),allow_unused=True)
    active=[g for g in grads if g is not None]
    if not active or not all(bool(torch.isfinite(g).all()) for g in active):
        raise RuntimeError(f"{name}: invalid or absent gradients")
    if sum(float(g.abs().sum()) for g in active)<=0:
        raise RuntimeError(f"{name}: zero gradients")

    # Equal-length inputs (no shape-dependent CuDNN algorithm change) isolate
    # whether arbitrary noise in MASKED tokens can influence valid predictions.
    torch.manual_seed(337)
    length,padded=281,320
    x=torch.randn(1,padded,model.hidden_size,device=device,dtype=torch.float32)
    mark=torch.zeros(1,padded,device=device,dtype=torch.long)
    mark[:,:length]=1
    clean=x.clone()
    noisy=x.clone()
    clean[:,length:]=0
    noisy[:,length:]=torch.randn_like(noisy[:,length:])*7
    model.eval()
    with torch.no_grad(),torch.autocast(device_type="cuda",enabled=False):
        a=model(clean,attention_mask=mark)[0]
        b=model(noisy,attention_mask=mark)[0]
    mask_diff=float((a[:,:length]-b[:,:length]).abs().max())
    if mask_diff>1e-4:
        raise RuntimeError(f"{name}: real masked-padding content leak Δ={mask_diff:.6g}")
    print(f" {name:<20} PASS | loss={float(loss):.5f} | gradients={len(active)}"
          f" | padding content Δ={mask_diff:.6g}")
    del model,grads,pred,loss
    torch.cuda.empty_cache()

for name in CANDIDATE_NAMES:
    preflight_model(name)

# Distinguish the content-routing hypothesis from U-Net's local information.
# Compare *untrained* compressed memory and U-Net on precisely the same probe batch.
untrained={}
for name in CANDIDATE_NAMES:
    mdl=build_candidate(name,teacher_attn).eval()
    with torch.no_grad(),torch.autocast(device_type="cuda",dtype=amp_dtype):
        p,_=mdl(sample["x"],attention_mask=sample["mask"],
                position_ids=sample["call_kwargs"].get("position_ids"))
    v=probe_first["attention_mask"].bool()
    untrained[name]=dict(cosine=float(F.cosine_similarity(
        p.float()[v],sample["y"].float()[v],dim=-1).mean()))
    print(" Untrained:",name,untrained[name])
    del mdl
torch.cuda.empty_cache()
print("Preflight passed; this is NOT proof any candidate meets fidelity thresholds.")


In [ ]:
#@title 6. Short equal-budget Q/K + U-Net teacher distillation, probe-only checkpoint selection
def state_cpu(module):
    return {k:v.detach().cpu().clone() for k,v in module.state_dict().items()}

def ranking_score(metrics):
    return metrics["nmse"]+1.10*(1.0-metrics["cosine"])

def cosine_lr(update,total,start,end):
    frac=min(update/max(total-1,1),1.)
    return end + 0.5*(start-end)*(1+math.cos(math.pi*frac))

def fit_local(name,model,training_cache,probe_cache,steps):
    params=candidate_parameters(model)
    conf=CANDIDATE_INFO[name]
    opt=torch.optim.AdamW(params,lr=conf["lr_start"],betas=(0.9,0.95),
                          weight_decay=1e-4)
    scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)
    best=local_metrics_from_cache(model,probe_cache)
    best_score=ranking_score(best)
    best_state=state_cpu(model)
    best_step=0
    history=[{"step":0,**best}]
    updates=attempts=0
    print(f"{name} initial: cos={best['cosine']:.5f} NMSE={best['nmse']:.5f}")
    while updates<steps:
        if attempts>steps+500: raise RuntimeError("Excessive nonfinite optimizer steps")
        entry=training_cache[attempts%len(training_cache)]
        lr=cosine_lr(updates,steps,conf["lr_start"],conf["lr_end"])
        for group in opt.param_groups: group["lr"]=lr
        model.train()
        model.out_drop.eval()  # deterministic frozen-teacher target
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda",dtype=amp_dtype):
            pred,_=model(entry["x"],attention_mask=entry["mask"],
                         position_ids=entry["call_kwargs"].get("position_ids"))
            valid=entry["valid"]
            p=pred.float()[valid]
            y=entry["y"].float()[valid]
            nmse=(p-y).square().sum()/y.square().sum().clamp_min(1e-8)
            cos=F.cosine_similarity(p,y,dim=-1).mean()
            loss=nmse+1.1*(1.0-cos)
        require_finite(loss,name+" loss")
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(params,1.0)
        scale_before=scaler.get_scale()
        scaler.step(opt)
        scaler.update()
        attempts+=1
        if scaler.is_enabled() and scaler.get_scale()<scale_before:
            continue
        updates+=1
        if updates==1 or updates%CHECK_EVERY==0 or updates==steps:
            model.eval()
            quality=local_metrics_from_cache(model,probe_cache)
            history.append({"step":updates,**quality})
            if ranking_score(quality)<best_score:
                best,best_score,best_step=quality,ranking_score(quality),updates
                best_state=state_cpu(model)
            print(f"  {updates:04d}/{steps}: probe cos={quality['cosine']:.4f} "
                  f"NMSE={quality['nmse']:.4f} | selected cosine={best['cosine']:.4f} @{best_step}")
    model.load_state_dict(best_state,strict=True)
    model.eval()
    return dict(updates=updates,attempts=attempts,steps=steps,
                best_step=best_step,probe=best,history=history)


In [ ]:
#@title 7. E2E teacher agreement and GPU latency of actual modules
,@torch.no_grad()
,def validate_pair(student, items, limit=None):
,    teacher.eval()
,    student.eval()
,    limit = len(items) if limit is None else min(int(limit), len(items))
,    n = matched = teacher_gold = student_gold = 0
,    jensen = []
,    for offset in range(0, limit, BATCH):
,        chosen = items[offset:min(offset+BATCH, limit)]
,        b = direct_batch(chosen)
,        with torch.autocast(device_type="cuda", dtype=amp_dtype):
,            tl, _ = teacher(**b)
,            sl, _ = student(**b)
,        require_finite(tl, "teacher logits")
,        require_finite(sl, "student logits")
,        mask = b["marker_mask"].bool()
,        tp = torch.softmax(tl.float().masked_fill(~mask, -1e9), dim=-1)
,        sp = torch.softmax(sl.float().masked_fill(~mask, -1e9), dim=-1)
,        gold = torch.tensor([c["gold_index"] for c in chosen], device=device)
,        n += len(chosen)
,        matched += int((tp.argmax(-1) == sp.argmax(-1)).sum())
,        teacher_gold += int((tp.argmax(-1) == gold).sum())
,        student_gold += int((sp.argmax(-1) == gold).sum())
,        mid = .5*(tp+sp)
,        j = .5*((tp*(tp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1))
,        j += .5*((sp*(sp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1))
,        jensen.extend(j.cpu().tolist())
,    return dict(decisions=n, agreement=matched/max(n,1), mean_js=float(np.mean(jensen)),
,                teacher_gold_accuracy=teacher_gold/max(n,1),
,                student_gold_accuracy=student_gold/max(n,1),
,                gold_accuracy_drop=(teacher_gold-student_gold)/max(n,1))
,
,@torch.inference_mode()
,def time_module(module, entry, *, native_teacher=False, warmup=5, repeats=20):
,    """Measure native SDPA with its captured encoder call, or a CeNN replacement.
,
,    ModernBERT SDPA requires position_ids and sliding_window_mask even when
,    only full attention is being benchmarked. Replaying the original call is
,    safer than synthesizing masks and prevents the missing-kwargs TypeError.
,    """
,    module.eval()
,    if native_teacher:
,        if "call_args" not in entry or "call_kwargs" not in entry:
,            raise RuntimeError(
,                "Teacher I/O cache predates SDPA signature fix. Rerun cell 3 "
,                "and rebuild cache in cell 8 (or recapture with attn_io)."
,            )
,        args, kwargs = entry["call_args"], entry["call_kwargs"]
,    else:
,        args = (entry["x"],)
,        kwargs = dict(position_ids=entry["call_kwargs"].get("position_ids"), attention_mask=entry["mask"])
,
,    def invoke():
,        return _attention_output_tensor(module(*args, **kwargs))
,
,    with torch.autocast(device_type="cuda", dtype=amp_dtype):
,        for _ in range(warmup):
,            last = invoke()
,        require_finite(last, "timing warmup output")
,        torch.cuda.synchronize()
,        times = []
,        for _ in range(repeats):
,            start = time.perf_counter()
,            invoke()
,            torch.cuda.synchronize()
,            times.append((time.perf_counter()-start)*1000)
,    return float(np.median(times))
,
,@torch.inference_mode()
,def time_model(model, chosen, warmup=3, repeats=10):
,    b = direct_batch(chosen)
,    model.eval()
,    with torch.autocast(device_type="cuda", dtype=amp_dtype):
,        for _ in range(warmup): model(**b)
,        torch.cuda.synchronize()
,        times=[]
,        for _ in range(repeats):
,            start=time.perf_counter()
,            model(**b)
,            torch.cuda.synchronize()
,            times.append((time.perf_counter()-start)*1000)
,    return float(np.median(times))
,
,def length_diagnostics(items):
,    lengths=np.array([len(it["ids"]) for it in items], dtype=np.int64)
,    return dict(items=len(items), median_tokens=float(np.median(lengths)),
,                p90_tokens=float(np.percentile(lengths,90)),
,                max_tokens=int(lengths.max()),
,                count_ge_256=int((lengths >= 256).sum()),
,                count_ge_512=int((lengths >= 512).sum()))
,
,@torch.no_grad()
,def evaluate_candidate(layer_idx, replacement, audit_cache, items=audit_items):
,    original = student_eval.encoder.layers[layer_idx].attn
,    student_eval.encoder.layers[layer_idx].attn = replacement
,    try:
,        replacement.eval()
,        local=local_metrics_from_cache(replacement, audit_cache)
,        downstream=validate_pair(student_eval, items)
,        entry=audit_cache[0]
,        teacher_ms=time_module(teacher.encoder.layers[layer_idx].attn, entry, native_teacher=True)
,        student_ms=time_module(replacement, entry, native_teacher=False)
,        fixed=items[:min(BATCH,len(items))]
,        teacher_total=time_model(teacher, fixed)
,        student_total=time_model(student_eval, fixed)
,        return dict(local=local, e2e=downstream, attention_ms=student_ms,
,                    teacher_attention_ms=teacher_ms,
,                    attention_speedup=teacher_ms/max(student_ms,1e-8),
,                    model_ms=student_total, teacher_model_ms=teacher_total,
,                    model_speedup=teacher_total/max(student_total,1e-8))
,    finally:
,        student_eval.encoder.layers[layer_idx].attn = original
,        student_eval.eval().requires_grad_(False)
,
,def gate(e):
,    local=e["local"]
,    downstream=e["e2e"]
,    return bool(local["cosine"] >= .95 and local["nmse"] <= .10
,                and downstream["agreement"] >= .95
,                and downstream["mean_js"] <= .02
,                and e["attention_speedup"] > 1.0)
,
,print("Probe length profile:", length_diagnostics(probe_items))
,print("Audit length profile:", length_diagnostics(audit_items))
,print("Teacher baseline:", validate_pair(student_eval, audit_items))


In [ ]:
#@title 8. Four-way SCREEN — safe checkpoint/resume, probe-selected, audit reported
# Cache identical frozen teacher examples once, avoiding repeated reruns.
train_cache=cache_teacher_io(LAYER,train_items,TRAIN_CACHE_BATCHES,offset=13)
probe_cache=cache_teacher_io(LAYER,probe_items,PROBE_CACHE_BATCHES,offset=0)
audit_cache=cache_teacher_io(LAYER,audit_items,AUDIT_CACHE_BATCHES,offset=0)
print("Unique cached batches:",len(train_cache),"train;",
      len(probe_cache),"probe;",len(audit_cache),"audit")
results=[]
for i,name in enumerate(CANDIDATE_NAMES):
    random.seed(SEED+101*i)
    np.random.seed(SEED+101*i)
    torch.manual_seed(SEED+101*i)
    torch.cuda.manual_seed_all(SEED+101*i)
    model=build_candidate(name,teacher.encoder.layers[LAYER].attn)
    path=OUTPUT_DIR/f"layer15_{name}_screen.pt"
    print("\n"+"="*86+"\n"+name.upper()+" — screen "+str(STEPS)+" updates\n"+"="*86)
    if path.exists():
        saved=torch.load(path,map_location="cpu",weights_only=False)
        if (saved.get("name")!=name or saved.get("steps")!=STEPS or
                saved.get("width")!=UNET_WIDTH or saved.get("stride")!=MEMORY_STRIDE):
            raise RuntimeError(f"Incompatible checkpoint {path}; change OUTPUT_DIR")
        model.load_state_dict(saved["state_dict"])
        trained=saved["train"]
        print("Resuming trained checkpoint",path)
    else:
        trained=fit_local(name,model,train_cache,probe_cache,STEPS)
        torch.save(dict(version="3.2",name=name,layer=LAYER,steps=STEPS,
                        width=UNET_WIDTH,stride=MEMORY_STRIDE,
                        architecture=model.config_dict(),
                        state_dict=state_cpu(model),train=trained),path)
        print("Saved before audit:",path)
    audit=evaluate_candidate(LAYER,model,audit_cache)
    rec=dict(name=name,train=trained,audit=audit,path=str(path),
             config=model.config_dict(),score=ranking_score(trained["probe"]),
             gates=gate(audit))
    results.append(rec)
    print(f"RESULT {name}: probe={trained['probe']['cosine']:.4f} "
          f"audit_cos={audit['local']['cosine']:.4f} "
          f"agree={audit['e2e']['agreement']:.3f} "
          f"accuracy_drop={audit['e2e']['gold_accuracy_drop']:+.3f} "
          f"layer_speedup={audit['attention_speedup']:.3f}x")
    del model
    torch.cuda.empty_cache()
ranked=sorted(results,key=lambda r:(r["score"],-r["train"]["probe"]["cosine"]))
winner=ranked[0]
print("\nProbe-selected:",winner["name"],"probe cosine",winner["train"]["probe"]["cosine"])
print("Note: audit was NOT used to select the winner.")


In [ ]:
#@title 9. OPTIONAL: refine only when probe performance justifies GPU cost
# No extra runs by default. Do NOT optimize based on audit performance.
refined_results=[]
if RUN_REFINE:
    go=[r for r in ranked if r["train"]["probe"]["cosine"] >= REFINE_GO_COSINE]
    if not go:
        print(f"STOP: all probe cosines < {REFINE_GO_COSINE:.2f}; no longer training.")
    else:
        for item in go[:REFINE_TOP_K]:
            name=item["name"]
            mdl=build_candidate(name,teacher.encoder.layers[LAYER].attn)
            saved=torch.load(item["path"],map_location="cpu",weights_only=False)
            mdl.load_state_dict(saved["state_dict"])
            ckpt=OUTPUT_DIR/f"layer15_{name}_refined.pt"
            if ckpt.exists():
                r=torch.load(ckpt,map_location="cpu",weights_only=False)
                mdl.load_state_dict(r["state_dict"])
                refinement=r["train"]
                print("Resumed refinement:",ckpt)
            else:
                refinement=fit_local(name,mdl,train_cache,probe_cache,REFINE_STEPS)
                torch.save(dict(version="3.2",name=name,layer=LAYER,
                                steps=STEPS+REFINE_STEPS,width=UNET_WIDTH,
                                stride=MEMORY_STRIDE,architecture=mdl.config_dict(),
                                state_dict=state_cpu(mdl),train=refinement),ckpt)
            audited=evaluate_candidate(LAYER,mdl,audit_cache)
            refined_results.append(dict(name=name,train=refinement,
                                        audit=audited,path=str(ckpt),
                                        gates=gate(audited)))
            print("Refined",name,"audit cosine",audited["local"]["cosine"])
            del mdl
            torch.cuda.empty_cache()
else:
    print("RUN_REFINE=False; no additional GPU training.")


In [ ]:
#@title 10. Report, plot, and stop/go interpretation
if not results: raise RuntimeError("Run the screen first")
def short(rec):
    a=rec["audit"]
    return (rec["name"],rec["train"]["probe"]["cosine"],
            a["local"]["cosine"],a["local"]["nmse"],
            a["e2e"]["agreement"],a["e2e"]["gold_accuracy_drop"],
            a["attention_speedup"])
print(f"{'Architecture':<22} {'Probe cos':>10} {'Audit cos':>10} "
      f"{'NMSE':>7} {'Agreement':>10} {'Acc drop':>9} {'Layer x':>8}")
for rec in results:
    name,pc,ac,nm,agree,drop,speed=short(rec)
    print(f"{name:<22} {pc:10.4f} {ac:10.4f} {nm:7.4f} "
          f"{agree:10.4f} {drop:9.3f} {speed:8.3f}")

best_by_probe=winner["name"]
ready=bool(winner["gates"])
if ready:
    print("\nPROMISING: winner passed the predeclared layer-15 gates; still validate on new seeds/lengths.")
else:
    print("\nNOT READY to replace full attention at layer 15.")
    if winner["train"]["probe"]["cosine"] < REFINE_GO_COSINE:
        print("Stop longer training: best probe cosine below GO threshold.")
    elif not winner["config"]["uses_pretrained_rope_qk"]:
        print("U-Net without Q/K may still lack selective remote retrieval.")
    else:
        print("Compressed landmark memory helps testing retrieval, but may lose key identities; "
              "consider smaller stride or richer positional/selective memory.")

report=dict(version="3.2",model=SOURCE_MODEL,layer=LAYER,
            seed=SEED,teacher_frozen=True,official_test_used=False,
            cases={"train":len(train_rows),"probe":len(probe_rows),"audit":len(audit_rows)},
            train_cached_decisions=len(train_cache)*BATCH,
            screen_updates=STEPS,refine_enabled=RUN_REFINE,
            memory_stride=MEMORY_STRIDE,unet_width=UNET_WIDTH,
            untrained_first_batch=untrained,
            winner_by_probe=best_by_probe,
            comparison=results,refined=refined_results,
            caveats=[
                "U-Net max-pooling/skip restoration is not equivalent to content-dependent self-attention",
                "Landmark Q/K memory includes compressed attention; it is NOT attention-free",
                "Pretrained QKV are frozen to distinguish routing from teacher retraining",
                "Using 3*ceil(T/stride) memory keys loses some original key identities",
                "The official test set is untouched; selected using probe, audit report-only",
                "Actual GPU latency is not implied by asymptotic score count",
            ])
p=OUTPUT_DIR/"v32_unet_maxpool_landmarks_report.json"
p.write_text(json.dumps(report,indent=2,allow_nan=False),encoding="utf-8")
print("Saved",p)
import matplotlib.pyplot as plt
for rec in results:
    h=rec["train"]["history"]
    plt.plot([e["step"] for e in h],[e["cosine"] for e in h],
             marker=".",label=rec["name"])
plt.axhline(.95,linestyle="--",label="0.95 acceptance cosine")
plt.xlabel("Optimizer updates")
plt.ylabel("Probe output cosine")
plt.title("U-Net pooling vs pretrained landmark Q/K retrieval")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
